# Football detection: YOLO26m @ 1280

- **Dataset v13** (1280×1280): bản độ phân giải cao nhất có đủ 372 ảnh.
- **Bỏ stretch**: Roboflow ép ảnh 16:9 thành hình vuông; resize về 1280×720 để giống lúc chạy thật.
- **yolo26m, imgsz=1280**: bóng chỉ vài pixel, cần độ phân giải cao.
- Weights lưu trên **Google Drive**, Colab có ngắt giữa chừng cũng không mất.

Runtime → Change runtime type → **GPU**.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/football_runs'

## GET DATASET
API key để trong **Colab Secrets** (biểu tượng 🔑 bên trái): tên `ROBOFLOW_API_KEY`, bật *Notebook access*.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(13).download("yolo26")
dataset.location

In [ ]:
import glob, cv2

# Roboflow stretched the 16:9 broadcast frames to 1280x1280 (players look tall and thin).
# Back to 1280x720 = what the model sees at inference (1920x1080 letterboxed to imgsz 1280).
# YOLO labels are normalized 0-1, so they stay valid as is.
images = glob.glob(f'{dataset.location}/*/images/*')
for path in images:
    img = cv2.imread(path)
    if img.shape[:2] != (720, 1280):
        cv2.imwrite(path, cv2.resize(img, (1280, 720), interpolation=cv2.INTER_AREA))
print(len(images), 'images ->', cv2.imread(images[0]).shape)

## Training

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26m.pt')  # 'yolo26s.pt' if training or inference on the Mac is too slow
model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=1280,
    batch=-1,    # AutoBatch: largest batch that fits the GPU
    cache=True,  # 372 images fit in RAM
    project=RUNS,
    name='yolo26m_1280',
)

In [ ]:
# Colab disconnected mid-training? Uncomment to resume from the last checkpoint on Drive
# (folder may be yolo26m_12802, ... if you trained more than once)
# model = YOLO(f'{RUNS}/yolo26m_1280/weights/last.pt'); model.train(resume=True)

## Evaluate
Model cũ (yolo26n @ 640, tập val): all mAP50 **0.769**, ball mAP50 **0.328** (recall 0.296).
Model này (yolo26m @ 1280, tập val): all mAP50 **0.931**, mAP50-95 **0.667**.

In [ ]:
best = YOLO(model.trainer.best)
best.val(data=f'{dataset.location}/data.yaml', split='test')

## Download
Chép `best.pt` vào `models/best.pt`, **xóa `stubs/track_stubs.pkl`**, rồi chạy `python main.py`.

In [ ]:
from google.colab import files
files.download(str(model.trainer.best))